# Desenvolvimento do Pipeline Medallion

A **Arquitetura Medallion** organiza nossos dados de e-commerce em três camadas lógicas de maturação (Bronze, Silver e Gold), garantindo governança, qualidade e estrutura incremental até o consumo pelo modelo preditivo.

## Ingestão na Camada Bronze (Dados Brutos e Confiáveis)

A **Camada Bronze** é o ponto de entrada do nosso Lakehouse. Ela armazena o histórico completo dos dados em seu formato original.

### Nosso Fluxo de Ingestão:
1. **Criação database**: Criar database para as camadas bronze, silver e gold.
2. **Upload no Volumes:** O arquivo bruto contendo o histórico de interações (`base_bronze_eventos.csv`) foi carregado diretamente em um **Volume do Unity Catalog**, nosso repositório seguro para arquivos não-estruturados.
3. **Persistência via PySpark SQL:** Utilizamos a interface de programação do Python para acionar comandos SQL de criação de tabela.
4. **Esquema Explícito (DDL Schema):** Em vez de forçar o cluster a adivinhar os tipos de dados (o que degrada a performance), declaramos manualmente o tipo estrito de cada coluna (`STRING`, `INT`, `DOUBLE`, `DECIMAL`).


### Dicionário de Dados da Tabela Bronze (`tabela_eventos`)

| Nome da Coluna | Tipo de Dado | Descrição / Propósito do Campo |
| :--- | :--- | :--- |
| **`id_evento`** | `STRING` | Identificador único universal (UUID) gerado para cada clique ou ação do usuário no e-commerce. |
| **`id_usuario`** | `STRING` | Código de identificação do cliente logado na plataforma |
| **`tipo_evento`** | `STRING` | Indica a ação realizada no funil de vendas (ex: `visualizar_produto`, `adicionar_ao_carrinho`, `finalizar_compra`). |
| **`timestamp_registro`** | `STRING` | Data e hora exata em que o evento ocorreu, registrado originalmente como string pelo sistema de tracking |
| **`dados_origem_json`** | `STRING` | Payload semiestruturado em formato de texto JSON. Contém as telemetrias avançadas e variáveis demográficas do clique, como dispositivo, sistema operacional, idade e renda do usuário. |

---

In [0]:
import pyspark

# Configuração de caminhos e nomes de tabela
catalogo_esquema_bronze = "workspace.bronze"

tabela_bronze = f"{catalogo_esquema_bronze}.tabela_eventos_bronze"

print("Iniciando a criação das tabelas de entrada na Camada Bronze...")

# -------------------------------------------------------------------------
# 1. CRIAÇÃO DA TABELA BRONZE (LOGS DE EVENTOS DO E-COMMERCE)
# -------------------------------------------------------------------------
print(f"Ingerindo arquivo de eventos para a tabela Delta: {tabela_bronze}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {tabela_bronze}
    USING DELTA
    AS 
    SELECT * FROM read_files(
      '/Volumes/workspace/bronze/raw/base_bronze_eventos.csv',
      format => 'csv',
      header => 'true',
      sep => ";",
      schema => '
        id_evento STRING,
        id_usuario STRING,
        tipo_evento STRING,
        timestamp_registro STRING,
        dados_origem_json STRING
      '
    )
""")

# -------------------------------------------------------------------------
# VALIDAÇÃO DOS DADOS INGERIDOS
# -------------------------------------------------------------------------
print("\n--- Amostra da Tabela Bronze (Eventos) ---")
display(spark.sql(f"SELECT * FROM {tabela_bronze}"))


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6248739765616593>, line 15
     10 # -------------------------------------------------------------------------
     11 # 1. CRIAÇÃO DA TABELA BRONZE (LOGS DE EVENTOS DO E-COMMERCE)
     12 # -------------------------------------------------------------------------
     13 print(f"Ingerindo arquivo de eventos para a tabela Delta: {tabela_bronze}")
---> 15 spark.sql(f"""
     16     CREATE OR REPLACE TABLE {tabela_bronze}
     17     USING DELTA
     18     AS 
     19     SELECT * FROM read_files(
     20       '/Volumes/workspace/bronze/raw/base_bronze_eventos.csv',
     21       format => 'csv',
     22       header => 'true',
     23       sep => ";",
     24       schema => '
     25         id_evento STRING,
     26         id_usuario STRING,
     27         tipo_evento STRING,
     28         timestamp_registro STRING

### Dicionário de Dados da Tabela de Dimensão (`dim_produtos`)

| Nome da Coluna | Tipo de Dado | Descrição / Propósito do Campo |
| :--- | :--- | :--- |
| **`id_produto`** | `STRING` | Chave primária da dimensão. Identificador exclusivo de cada produto do catálogo (ex: `prod_101`). |
| **`nome_produto`** | `STRING` | Nome comercial completo do item disponível na plataforma de e-commerce. |
| **`categoria`** | `STRING` | Agrupamento lógico de mercado ao qual o produto pertence (ex: `Eletrônicos`, `Vestuário`, `Calçados`). |
| **`preco_reais`** | `DECIMAL(18,2)` | Valor monetário sugerido do produto em Reais (R$). |

---

In [0]:
# -------------------------------------------------------------------------
# CRIAÇÃO DA TABELA DE DIMENSÃO (CADASTRO DE PRODUTOS)
# -------------------------------------------------------------------------

tabela_produtos = f"{catalogo_esquema_bronze}.dim_produtos"
print(f" Ingerindo cadastro de produtos para a tabela Delta: {tabela_produtos}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {tabela_produtos}
    USING DELTA
    AS 
    SELECT * FROM read_files(
      '/Volumes/workspace/bronze/raw/base_dim_produtos.csv',
      format => 'csv',
      header => 'true',
      schema => '
        id_produto STRING,
        nome_produto STRING,
        categoria STRING,
        preco_reais DECIMAL(18,2)
      '
    )
""")

print(" Ingestão concluída com sucesso!")


# -------------------------------------------------------------------------
# VALIDAÇÃO DOS DADOS INGERIDOS
# -------------------------------------------------------------------------
print("\n--- Amostra da Tabela de Dimensão (Produtos) ---")
display(spark.sql(f"SELECT * FROM {tabela_produtos}"))

## Construção da Camada Silver (Limpeza e Estruturação)

Nesta etapa, nosso objetivo é realizar o **ETL clássico** sobre os logs brutos da Camada Bronze. O Spark irá processar o conjunto de dados para:
1. **Remover duplicatas:** Eliminar registros duplicados idênticos causados por falhas ou cliques duplos.
2. **Explodir o JSON:** Utilizar a função `from_json` para quebrar a string de texto bruto em colunas relacionais estruturadas (`id_produto`, `dispositivo`, `ip_cliente`).
3. **Tipagem estrita:** Tipar os campos de forma otimizada para consultas.

In [0]:
import pyspark

catalogo_esquema_silver = "workspace.silver"
tabela_silver = f"{catalogo_esquema_silver}.tabela_eventos_silver"

print(f"Processando a Camada Silver ({tabela_silver}) a partir dos dados do JSON...")

# Execução do pipeline via Spark SQL
spark.sql(f"""
CREATE OR REPLACE TABLE {tabela_silver}
    USING DELTA
    AS
    WITH dados_deduplicados AS (
        SELECT DISTINCT 
            id_evento,
            id_usuario,
            tipo_evento,
            CAST(timestamp_registro AS TIMESTAMP) AS timestamp_registro,
            
            -- LIMPEZA DO JSON (Tratamento para remover aspas duplicadas corporativas do CSV)
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    TRIM(dados_origem_json), 
                    '^"|"$', '' -- Remove as aspas que envelopam o texto nas pontas
                ), 
                '""', '"'      -- Transforma duas aspas "" em apenas uma "
            ) AS json_limpo
            
        FROM {tabela_bronze}
    ),
    dados_parseados AS (
        SELECT 
            id_evento,
            id_usuario,
            tipo_evento,
            timestamp_registro,
            
            -- Faz o parse em cima do campo texto que agora está perfeitamente higienizado
            from_json(
                json_limpo, 
                'id_produto STRING, 
                 dispositivo STRING, 
                 sistema_operacional STRING, 
                 origem_trafego STRING, 
                 idade INT, 
                 genero STRING, 
                 renda_mensal_estimada_k DECIMAL(18,2), 
                 regiao STRING, 
                 tempo_clique_segundos INT, 
                 interacoes_chat_suporte INT, 
                 cupons_ativos_conta INT, 
                 score_satisfacao_nps INT, 
                 dias_desde_ultima_visita INT'
            ) AS payload
        FROM dados_deduplicados
    )
    -- Projeção final espalhando as propriedades nas colunas relacionais
    SELECT 
        id_evento,
        id_usuario,
        tipo_evento,
        timestamp_registro,
        payload.id_produto,
        payload.dispositivo,
        payload.sistema_operacional,
        payload.origem_trafego,
        payload.idade,
        payload.genero,
        payload.renda_mensal_estimada_k,
        payload.regiao,
        payload.tempo_clique_segundos,
        payload.interacoes_chat_suporte,
        payload.cupons_ativos_conta,
        payload.score_satisfacao_nps,
        payload.dias_desde_ultima_visita
    FROM dados_parseados
""")

print("✅ Camada Silver gerada com sucesso com colunas totalmente estruturadas!")

# Visualizando o resultado final com as colunas separadas
display(spark.sql(f"SELECT * FROM {tabela_silver}"))

In [0]:
display(spark.sql(f"SELECT * FROM {tabela_silver}"))

%md
## Construção da Camada Gold

Nesta etapa vamos fazer um union com a tabela de produtos `dim_produtos` com base no `id_produto` para ter em uma tabela final na tabela final as informações de eventos da camada silver e o `nome_produto`, `categoria` e  `preco_reais`.

In [0]:
import pyspark

catalogo_esquema = "workspace.gold"
tabela_gold = f"{catalogo_esquema}.base_eventos_produto_gold"

# Execução do pipeline via Spark SQL
spark.sql(f"""
CREATE OR REPLACE TABLE {tabela_gold}
    USING DELTA
    AS
        -- Cruzamento da Silver com a Dimensão de Produtos
        SELECT 
            s.id_usuario,
            s.tipo_evento,
            s.dispositivo,
            s.sistema_operacional,
            s.origem_trafego,
            s.idade,
            s.genero,
            s.renda_mensal_estimada_k,
            s.regiao,
            s.tempo_clique_segundos,
            s.interacoes_chat_suporte,
            s.cupons_ativos_conta,
            s.score_satisfacao_nps,
            s.dias_desde_ultima_visita,
            p.categoria,
            p.preco_reais
        FROM {tabela_silver} s
        INNER JOIN {tabela_produtos} p ON s.id_produto = p.id_produto
""")

print("Camada Gold consolidada com sucesso!")
display(spark.sql(f"SELECT * FROM {tabela_gold}"))

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# 1. Carregar a base Gold direto do Unity Catalog
tabela_gold = "workspace.gold.base_modelo_preditivo_gold"
df_gold = spark.table(tabela_gold)

# Definindo a coluna alvo (Target)
target_col = "conversao_futura_eletronicos"

# Separando as variáveis numéricas e categóricas que vão alimentar o modelo
num_features = [
    "idade_cliente", "renda_mensal_estimada_k", "tempo_medio_clique_segundos", 
    "media_interacoes_suporte", "cupons_ativos", "score_nps_cliente", 
    "dias_desde_ultima_visita", "total_visualizacoes", "total_adicoes_carrinho", "total_compras_gerais"
]
cat_features = ["genero_cliente", "regiao_cliente", "dispositivo_mais_usado", "so_mais_usado", "canal_origem_trafego", "categoria_mais_visitada"]

print("🛠️ Construindo o pipeline de engenharia de atributos...")

# 2. Pipeline de Transformação de Texto para as Variáveis Categóricas
stages = []
for col in cat_features:
    indexer = StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    stages += [indexer, encoder]

# Lista com todas as colunas prontas após o encoding
encoded_cat_cols = [f"{col}_encoded" for col in cat_features]
all_features_cols = num_features + encoded_cat_cols

# 3. Juntando todas as variáveis explicativas em um único vetor (Vector Assembler)
assembler = VectorAssembler(inputCols=all_features_cols, outputCol="features")
stages.append(assembler)

# 4. Instanciando o Modelo Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol=target_col, numTrees=50, maxDepth=8, seed=42)
stages.append(rf)

# Criando o Pipeline completo do Spark ML
pipeline_ml = Pipeline(stages=stages)

# 5. Divisão Amostral (70% Treino / 30% Teste)
print("📊 Dividindo a base Gold em Treino (70%) e Teste (30%)...")
df_treino, df_teste = df_gold.randomSplit([0.7, 0.3], seed=42)

# 6. Treinando o Modelo
print("🚀 Treinando o classificador Random Forest no cluster Databricks...")
modelo_treinado = pipeline_ml.fit(df_treino)

# 7. Gerando as Predições na Base de Teste
print("🔍 Aplicando o modelo na base de teste para validação...")
df_predicoes = modelo_treinado.transform(df_teste)

# -------------------------------------------------------------------------
# 8. AVALIAÇÃO DE MÉTRICAS (Métricas de Sucesso)
# -------------------------------------------------------------------------
print("\n" + "="*40)
print("📈 RESULTADOS DE PERFORMANCE DO MODELO")
print("="*40)

# Calculando a Acurácia Geral
evaluator_acc = MulticlassClassificationEvaluator(labelCol=target_col, predictionCol="prediction", metricName="accuracy")
acuracia = evaluator_acc.evaluate(df_predicoes)
print(f"✅ Acurácia do Modelo: {acuracia * 100:.2f}%")

# Calculando a Área sob a Curva ROC (AUC-ROC)
evaluator_auc = BinaryClassificationEvaluator(labelCol=target_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_roc = evaluator_auc.evaluate(df_predicoes)